In [1]:
# Cell 0 — Project Imports

import torch

In [2]:
# Cell 1 — Batch-Slot Foreground Oversampling Decision

def determine_force_foreground(
    batch_index: int,
    batch_size: int,
    oversample_foreground_fraction: float,
) -> tuple[
    bool,  # 현재 batch slot의 foreground 강제 여부
    int,   # 첫 번째 foreground 강제 slot index
]:
    """Deterministic batch-slot foreground oversampling 결정."""

    # 0 이하의 잘못된 batch size 차단
    if batch_size <= 0:
        raise ValueError(
            "batch_size는 양수여야 합니다."
        )

    # Batch 범위를 벗어난 slot index 차단
    if not 0 <= batch_index < batch_size:
        raise ValueError(
            "batch_index는 0 이상 batch_size 미만이어야 합니다."
        )

    # 확률 범위를 벗어난 oversampling fraction 차단
    if not 0.0 <= oversample_foreground_fraction <= 1.0:
        raise ValueError(
            "oversampling fraction은 0.0 이상 1.0 이하여야 합니다."
        )

    # Batch 앞쪽에서 foreground를 강제하지 않을 slot 개수 계산
    first_forced_foreground_index = round(
        batch_size
        * (
            1.0
            - oversample_foreground_fraction
        )
    )

    # 경계 index 이후의 batch slot을 foreground 경로로 지정
    force_foreground = (
        batch_index
        >= first_forced_foreground_index
    )

    return (
        force_foreground,
        first_forced_foreground_index,
    )


# nnU-Net 기본값에 대응하는 foreground oversampling fraction 설정
oversample_foreground_fraction = 0.33


# 작은 batch size에서 반올림으로 달라지는 실제 forced 비율 확인
for batch_size in (
    2,
    3,
    4,
    5,
):
    forced_slot_indices: list[int] = []
    first_forced_index = 0

    # 현재 batch의 모든 slot을 순서대로 검사
    for batch_index in range(
        batch_size
    ):
        (
            force_foreground,
            first_forced_index,
        ) = determine_force_foreground(
            batch_index=batch_index,
            batch_size=batch_size,
            oversample_foreground_fraction=(
                oversample_foreground_fraction
            ),
        )

        # Foreground 경로로 지정된 slot index 저장
        if force_foreground:
            forced_slot_indices.append(
                batch_index
            )

    # 반올림 이후 실제 batch에서 구현된 forced 비율 계산
    realized_forced_fraction = (
        len(forced_slot_indices)
        / batch_size
    )

    print(
        f"Batch size={batch_size} | "
        f"first forced index={first_forced_index} | "
        f"forced slots={forced_slot_indices} | "
        f"realized fraction={realized_forced_fraction:.3f}"
    )


# Batch size 2의 slot별 결정 과정 상세 출력
example_batch_size = 2

print()
print("Batch size 2 decision trace")

for batch_index in range(
    example_batch_size
):
    (
        force_foreground,
        first_forced_index,
    ) = determine_force_foreground(
        batch_index=batch_index,
        batch_size=example_batch_size,
        oversample_foreground_fraction=(
            oversample_foreground_fraction
        ),
    )

    sampling_route = (
        "foreground"
        if force_foreground
        else "random"
    )

    print(
        f"slot={batch_index} | "
        f"boundary={first_forced_index} | "
        f"force_fg={force_foreground} | "
        f"route={sampling_route}"
    )

Batch size=2 | first forced index=1 | forced slots=[1] | realized fraction=0.500
Batch size=3 | first forced index=2 | forced slots=[2] | realized fraction=0.333
Batch size=4 | first forced index=3 | forced slots=[3] | realized fraction=0.250
Batch size=5 | first forced index=3 | forced slots=[3, 4] | realized fraction=0.400

Batch size 2 decision trace
slot=0 | boundary=1 | force_fg=False | route=random
slot=1 | boundary=1 | force_fg=True | route=foreground


In [3]:
# Cell 2 — Case Selection과 force_fg 분리

def sample_case_keys(
    available_case_keys: tuple[str, ...],
    batch_size: int,
    generator: torch.Generator,
) -> tuple[str, ...]:
    """Replacement를 허용한 uniform case key 선택."""

    # 비어 있는 Dataset 차단
    if len(available_case_keys) == 0:
        raise ValueError(
            "선택 가능한 case가 하나 이상 필요합니다."
        )

    # 잘못된 batch size 차단
    if batch_size <= 0:
        raise ValueError(
            "batch_size는 양수여야 합니다."
        )

    # 각 batch slot에서 사용할 case index를 독립적으로 선택
    sampled_case_indices = torch.randint(
        low=0,
        high=len(available_case_keys),
        size=(batch_size,),
        generator=generator,
    )  # [B]

    # 정수 index를 실제 case key로 변환
    sampled_case_keys = tuple(
        available_case_keys[
            case_index
        ]
        for case_index in sampled_case_indices.tolist()
    )

    return sampled_case_keys


# 선택 가능한 가상 환자 목록 구성
available_case_keys: tuple[str, ...] = (
    "Case_A",
    "Case_B",
    "Case_C",
)


# 재현 가능한 case sampling을 위한 난수 생성기 구성
case_sampling_generator = torch.Generator()
case_sampling_generator.manual_seed(
    55254
)


# 작은 GPU 상황을 가정한 batch size 설정
batch_size = 2
oversample_foreground_fraction = 0.33


# 세 개의 연속 training batch 생성
for batch_number in range(3):
    sampled_case_keys = sample_case_keys(
        available_case_keys=available_case_keys,
        batch_size=batch_size,
        generator=case_sampling_generator,
    )

    print(
        f"Batch {batch_number}"
    )

    # Case 선택 결과와 batch-slot sampling 경로 결합
    for (
        batch_index,
        sampled_case_key,
    ) in enumerate(
        sampled_case_keys
    ):
        (
            force_foreground,
            _,
        ) = determine_force_foreground(
            batch_index=batch_index,
            batch_size=batch_size,
            oversample_foreground_fraction=(
                oversample_foreground_fraction
            ),
        )

        sampling_route = (
            "foreground"
            if force_foreground
            else "random"
        )

        print(
            f"  slot={batch_index} | "
            f"case={sampled_case_key} | "
            f"force_fg={force_foreground} | "
            f"route={sampling_route}"
        )

Batch 0
  slot=0 | case=Case_A | force_fg=False | route=random
  slot=1 | case=Case_B | force_fg=True | route=foreground
Batch 1
  slot=0 | case=Case_C | force_fg=False | route=random
  slot=1 | case=Case_C | force_fg=True | route=foreground
Batch 2
  slot=0 | case=Case_B | force_fg=False | route=random
  slot=1 | case=Case_C | force_fg=True | route=foreground


In [4]:
# Cell 3 — Class와 Annotated Location 선택

def sample_foreground_class_and_location(
    class_locations: dict[
        int,
        torch.Tensor,  # Class별 foreground 좌표 [V_k, 3]
    ],
    generator: torch.Generator,
) -> tuple[
    int,           # 선택된 class ID
    torch.Tensor,  # 선택된 foreground 좌표 [3]
]:
    """Eligible class와 class 내부 foreground voxel 좌표 선택."""

    # Class location 정보가 없는 잘못된 입력 차단
    if len(class_locations) == 0:
        raise ValueError(
            "class_locations가 하나 이상 필요합니다."
        )

    eligible_class_ids: list[int] = []

    # 각 class의 coordinate Tensor Shape와 dtype 확인
    for (
        class_id,
        voxel_locations_zyx,
    ) in class_locations.items():
        if (
            voxel_locations_zyx.ndim != 2
            or voxel_locations_zyx.shape[1] != 3
        ):
            raise ValueError(
                f"class {class_id} locations의 Shape는 [V_k, 3]이어야 합니다."
            )

        if torch.is_floating_point(
            voxel_locations_zyx
        ):
            raise TypeError(
                f"class {class_id} locations는 integer Tensor여야 합니다."
            )

        # Foreground voxel이 하나 이상 존재하는 class만 선택 후보에 포함
        if voxel_locations_zyx.shape[0] > 0:
            eligible_class_ids.append(
                class_id
            )

    # 현재 환자에 foreground가 하나도 없는 상황 차단
    if len(eligible_class_ids) == 0:
        raise ValueError(
            "선택 가능한 foreground class가 없습니다."
        )

    # Eligible class 목록에서 하나를 균일하게 선택
    selected_class_list_index = int(
        torch.randint(
            low=0,
            high=len(eligible_class_ids),
            size=(1,),
            generator=generator,
        ).item()
    )

    selected_class_id = eligible_class_ids[
        selected_class_list_index
    ]

    selected_class_locations = class_locations[
        selected_class_id
    ]  # [V_selected, 3]

    # 선택된 class 내부의 annotated voxel 중 하나를 균일하게 선택
    selected_voxel_index = int(
        torch.randint(
            low=0,
            high=selected_class_locations.shape[0],
            size=(1,),
            generator=generator,
        ).item()
    )

    selected_location_zyx = (
        selected_class_locations[
            selected_voxel_index
        ].clone()
    )  # [3]

    return (
        selected_class_id,
        selected_location_zyx,
    )


# 큰 장기 class 1의 가상 foreground 좌표 구성
large_organ_locations = torch.tensor(
    [
        [10, 20, 20],
        [10, 20, 21],
        [10, 21, 20],
        [10, 21, 21],
        [11, 20, 20],
        [11, 20, 21],
        [11, 21, 20],
        [11, 21, 21],
    ],
    dtype=torch.int64,
)  # [V_1=8, 3]


# 작은 장기 class 2의 가상 foreground 좌표 구성
small_organ_locations = torch.tensor(
    [
        [24, 40, 40],
        [24, 40, 41],
    ],
    dtype=torch.int64,
)  # [V_2=2, 3]


# 현재 환자에 존재하지 않는 class 3의 빈 좌표 구성
absent_organ_locations = torch.empty(
    0,
    3,
    dtype=torch.int64,
)  # [V_3=0, 3]


# Preprocessing에서 저장됐다고 가정한 class별 좌표 dictionary 구성
class_locations: dict[
    int,
    torch.Tensor,
] = {
    1: large_organ_locations,
    2: small_organ_locations,
    3: absent_organ_locations,
}


# 재현 가능한 class·voxel 선택을 위한 난수 생성기 구성
foreground_generator = torch.Generator()
foreground_generator.manual_seed(
    55254
)


# 한 번의 foreground sampling 결과 확인
(
    selected_class_id,
    selected_location_zyx,
) = sample_foreground_class_and_location(
    class_locations=class_locations,
    generator=foreground_generator,
)

print(
    "Selected class:",
    selected_class_id,
)

print(
    "Selected location:",
    selected_location_zyx.tolist(),
)


# Class별 좌표 개수와 실제 선택 빈도의 관계 확인
number_of_sampling_trials = 2000

class_selection_counts: dict[int, int] = {
    1: 0,
    2: 0,
    3: 0,
}

for _ in range(
    number_of_sampling_trials
):
    (
        sampled_class_id,
        _,
    ) = sample_foreground_class_and_location(
        class_locations=class_locations,
        generator=foreground_generator,
    )

    class_selection_counts[
        sampled_class_id
    ] += 1


print()
print("Class selection statistics")

for class_id in sorted(
    class_locations
):
    voxel_count = class_locations[
        class_id
    ].shape[0]

    selection_count = class_selection_counts[
        class_id
    ]

    selection_fraction = (
        selection_count
        / number_of_sampling_trials
    )

    print(
        f"class={class_id} | "
        f"foreground voxels={voxel_count} | "
        f"selected={selection_count} | "
        f"fraction={selection_fraction:.3f}"
    )

Selected class: 2
Selected location: [24, 40, 41]

Class selection statistics
class=1 | foreground voxels=8 | selected=981 | fraction=0.490
class=2 | foreground voxels=2 | selected=1019 | fraction=0.509
class=3 | foreground voxels=0 | selected=0 | fraction=0.000


In [5]:
# Cell 4 — Foreground Bounding Box와 Patch Crop

def compute_forced_foreground_bbox(
    volume_shape_zyx: torch.Tensor,       # [3], integer
    patch_size_zyx: torch.Tensor,         # [3], integer
    selected_location_zyx: torch.Tensor,  # [3], integer
) -> tuple[
    torch.Tensor,  # Bounding-box lower bounds [3]
    torch.Tensor,  # Bounding-box upper bounds [3]
]:
    """선택된 foreground voxel을 포함하는 patch bounding box 계산."""

    # 모든 geometry가 (z, y, x) 세 axis를 가지는지 확인
    if (
        volume_shape_zyx.shape != (3,)
        or patch_size_zyx.shape != (3,)
        or selected_location_zyx.shape != (3,)
    ):
        raise ValueError(
            "모든 geometry Tensor의 Shape는 [3]이어야 합니다."
        )

    # 0 이하의 volume 또는 patch 크기 차단
    if not bool(
        torch.all(volume_shape_zyx > 0).item()
        and torch.all(patch_size_zyx > 0).item()
    ):
        raise ValueError(
            "Volume Shape와 patch size는 모두 양수여야 합니다."
        )

    # Volume 범위를 벗어난 foreground 좌표 차단
    location_is_inside = torch.all(
        (
            selected_location_zyx >= 0
        )
        & (
            selected_location_zyx
            < volume_shape_zyx
        )
    )

    if not bool(
        location_is_inside.item()
    ):
        raise ValueError(
            "선택된 foreground 좌표가 volume 내부에 있어야 합니다."
        )

    # Patch가 volume보다 큰 axis에서 필요한 총 padding 크기 계산
    need_to_pad_zyx = torch.clamp(
        patch_size_zyx
        - volume_shape_zyx,
        min=0,
    )  # [3]

    # 필요한 padding을 양쪽으로 분배한 최소 lower bound 계산
    minimum_lower_bounds_zyx = torch.div(
        -need_to_pad_zyx,
        2,
        rounding_mode="floor",
    )  # [3]

    # 선택된 foreground voxel을 patch 중앙에 두는 lower bound 계산
    desired_lower_bounds_zyx = (
        selected_location_zyx
        - torch.div(
            patch_size_zyx,
            2,
            rounding_mode="floor",
        )
    )  # [3]

    # 허용 범위보다 낮은 bounding-box 시작점 차단
    bbox_lower_bounds_zyx = torch.maximum(
        desired_lower_bounds_zyx,
        minimum_lower_bounds_zyx,
    )  # [3]

    # Half-open interval의 upper bound 계산
    bbox_upper_bounds_zyx = (
        bbox_lower_bounds_zyx
        + patch_size_zyx
    )  # [3]

    return (
        bbox_lower_bounds_zyx,
        bbox_upper_bounds_zyx,
    )


def crop_and_pad_label_patch(
    label_volume: torch.Tensor,       # [C, D, H, W], torch.long
    bbox_lower_zyx: torch.Tensor,     # [3], integer
    bbox_upper_zyx: torch.Tensor,     # [3], integer
    padding_value: int = -1,
) -> torch.Tensor:                    # [C, pD, pH, pW]
    """Bounding box의 유효 영역 crop과 외부 영역 padding."""

    # Channel과 세 spatial axis를 가진 label인지 확인
    if label_volume.ndim != 4:
        raise ValueError(
            "label_volume의 Shape는 [C, D, H, W]여야 합니다."
        )

    if label_volume.dtype != torch.long:
        raise TypeError(
            "label_volume은 torch.long Tensor여야 합니다."
        )

    # Label volume의 spatial Shape 추출
    volume_shape_zyx = torch.tensor(
        label_volume.shape[-3:],
        dtype=torch.int64,
    )  # [3]

    # Bounding box와 volume이 겹치는 실제 crop 범위 계산
    clipped_lower_zyx = torch.maximum(
        bbox_lower_zyx,
        torch.zeros_like(
            bbox_lower_zyx
        ),
    )  # [3]

    clipped_upper_zyx = torch.minimum(
        bbox_upper_zyx,
        volume_shape_zyx,
    )  # [3]

    (
        lower_z,
        lower_y,
        lower_x,
    ) = clipped_lower_zyx.tolist()

    (
        upper_z,
        upper_y,
        upper_x,
    ) = clipped_upper_zyx.tolist()

    # Volume 내부에 존재하는 label 영역만 crop
    cropped_label = label_volume[
        :,
        lower_z:upper_z,
        lower_y:upper_y,
        lower_x:upper_x,
    ]  # [C, cropped_D, cropped_H, cropped_W]

    # Bounding box가 volume 시작점 밖으로 나간 크기 계산
    padding_before_zyx = (
        clipped_lower_zyx
        - bbox_lower_zyx
    )  # [3]

    # Bounding box가 volume 끝점 밖으로 나간 크기 계산
    padding_after_zyx = (
        bbox_upper_zyx
        - clipped_upper_zyx
    )  # [3]

    (
        padding_before_z,
        padding_before_y,
        padding_before_x,
    ) = padding_before_zyx.tolist()

    (
        padding_after_z,
        padding_after_y,
        padding_after_x,
    ) = padding_after_zyx.tolist()

    # PyTorch가 요구하는 역순 (x, y, z) padding tuple 구성
    padding_xyz = (
        padding_before_x,
        padding_after_x,
        padding_before_y,
        padding_after_y,
        padding_before_z,
        padding_after_z,
    )

    # Volume 외부 영역을 ignore label 값으로 padding
    padded_label_patch = torch.nn.functional.pad(
        cropped_label,
        pad=padding_xyz,
        mode="constant",
        value=padding_value,
    )  # [C, pD, pH, pW]

    return padded_label_patch


# 선택된 class location을 포함할 가상 label volume 생성
synthetic_label_volume = torch.zeros(
    1,
    32,
    64,
    64,
    dtype=torch.long,
)  # [C=1, D=32, H=64, W=64]


# Class 1 foreground 좌표를 label volume에 기록
for coordinate_zyx in large_organ_locations:
    (
        coordinate_z,
        coordinate_y,
        coordinate_x,
    ) = coordinate_zyx.tolist()

    synthetic_label_volume[
        0,
        coordinate_z,
        coordinate_y,
        coordinate_x,
    ] = 1


# Class 2 foreground 좌표를 label volume에 기록
for coordinate_zyx in small_organ_locations:
    (
        coordinate_z,
        coordinate_y,
        coordinate_x,
    ) = coordinate_zyx.tolist()

    synthetic_label_volume[
        0,
        coordinate_z,
        coordinate_y,
        coordinate_x,
    ] = 2


# Data loader가 반환해야 할 고정 patch 크기 설정
patch_size_zyx = torch.tensor(
    [8, 16, 16],
    dtype=torch.int64,
)  # [3]


# Cell 3에서 선택한 foreground 좌표의 bounding box 계산
(
    bbox_lower_zyx,
    bbox_upper_zyx,
) = compute_forced_foreground_bbox(
    volume_shape_zyx=torch.tensor(
        synthetic_label_volume.shape[-3:],
        dtype=torch.int64,
    ),
    patch_size_zyx=patch_size_zyx,
    selected_location_zyx=selected_location_zyx,
)


# Bounding box 영역을 고정 Shape patch로 crop·padding
sampled_label_patch = crop_and_pad_label_patch(
    label_volume=synthetic_label_volume,
    bbox_lower_zyx=bbox_lower_zyx,
    bbox_upper_zyx=bbox_upper_zyx,
    padding_value=-1,
)  # [C=1, pD=8, pH=16, pW=16]


# 원본 foreground 좌표를 patch 내부 local 좌표로 변환
selected_local_location_zyx = (
    selected_location_zyx
    - bbox_lower_zyx
)  # [3]

(
    local_z,
    local_y,
    local_x,
) = selected_local_location_zyx.tolist()


# 선택한 foreground class가 실제 patch 안에 포함됐는지 확인
selected_patch_class_id = int(
    sampled_label_patch[
        0,
        local_z,
        local_y,
        local_x,
    ].item()
)


print(
    "Selected class:",
    selected_class_id,
)

print(
    "Selected global location:",
    selected_location_zyx.tolist(),
)

print(
    "Bounding-box lower:",
    bbox_lower_zyx.tolist(),
)

print(
    "Bounding-box upper:",
    bbox_upper_zyx.tolist(),
)

print(
    "Patch Shape:",
    list(
        sampled_label_patch.shape
    ),
)

print(
    "Selected local location:",
    selected_local_location_zyx.tolist(),
)

print(
    "Class ID inside patch:",
    selected_patch_class_id,
)

print(
    "Selected class preserved:",
    selected_patch_class_id
    == selected_class_id,
)

Selected class: 2
Selected global location: [24, 40, 41]
Bounding-box lower: [20, 32, 33]
Bounding-box upper: [28, 48, 49]
Patch Shape: [1, 8, 16, 16]
Selected local location: [4, 8, 8]
Class ID inside patch: 2
Selected class preserved: True


In [6]:
# Cell 5 — Default·Static·OLES3D Sampling Boundary

def sample_weighted_pools(
    pool_names: tuple[str, ...],
    pool_weights: torch.Tensor,  # [P], floating-point
    number_of_samples: int,
    generator: torch.Generator,
) -> dict[str, int]:
    """동일 candidate pool에서 weight 기반 선택 횟수 계산."""

    # Pool contract와 weight 유효성 확인
    if pool_weights.shape != (len(pool_names),):
        raise ValueError("pool_weights의 Shape는 [P]여야 합니다.")
    if not torch.is_floating_point(pool_weights):
        raise TypeError("pool_weights는 floating-point Tensor여야 합니다.")
    if bool(torch.any(pool_weights < 0).item()) or float(pool_weights.sum().item()) <= 0.0:
        raise ValueError("Pool weight는 음수가 아니며 하나 이상 양수여야 합니다.")
    if number_of_samples <= 0:
        raise ValueError("number_of_samples는 양수여야 합니다.")

    # Weight에 비례한 pool index sampling
    sampled_indices = torch.multinomial(
        pool_weights, number_of_samples, replacement=True, generator=generator
    )  # [S]

    # Pool별 선택 횟수 집계
    counts = {pool_name: 0 for pool_name in pool_names}
    for pool_index in sampled_indices.tolist():
        counts[pool_names[pool_index]] += 1
    return counts


# Static과 OLES3D가 공유할 동일한 semantic candidate pool
semantic_pool_names = (
    "interior_miss",
    "boundary_disagreement",
    "exterior_false_positive",
)
static_weights = torch.tensor([1.0, 1.0, 1.0], dtype=torch.float32)  # [P=3]
adaptive_weights = torch.tensor([1.0, 4.0, 2.0], dtype=torch.float32)  # [P=3]

# 동일 seed와 candidate pool에서 selection weight만 변경
static_generator = torch.Generator().manual_seed(55254)
adaptive_generator = torch.Generator().manual_seed(55254)
number_of_samples = 3000
static_counts = sample_weighted_pools(
    semantic_pool_names, static_weights, number_of_samples, static_generator
)
adaptive_counts = sample_weighted_pools(
    semantic_pool_names, adaptive_weights, number_of_samples, adaptive_generator
)

# 세 comparator의 변경 경계 명시
sampling_contracts = {
    "nnunet_default": "eligible class -> annotated voxel",
    "matched_static": "same semantic pools + fixed weights",
    "oles3d_adaptive": "same semantic pools + adaptive weights",
}
for sampler_name, contract in sampling_contracts.items():
    print(f"{sampler_name:16s} | {contract}")

print()
for pool_index, pool_name in enumerate(semantic_pool_names):
    static_fraction = static_counts[pool_name] / number_of_samples
    adaptive_fraction = adaptive_counts[pool_name] / number_of_samples
    print(
        f"{pool_name:24s} | static={static_fraction:.3f} | "
        f"adaptive={adaptive_fraction:.3f}"
    )

print("Same candidate pools:", tuple(static_counts) == tuple(adaptive_counts))
print("Only weights differ:", not torch.equal(static_weights, adaptive_weights))

nnunet_default   | eligible class -> annotated voxel
matched_static   | same semantic pools + fixed weights
oles3d_adaptive  | same semantic pools + adaptive weights

interior_miss            | static=0.326 | adaptive=0.141
boundary_disagreement    | static=0.343 | adaptive=0.581
exterior_false_positive  | static=0.331 | adaptive=0.278
Same candidate pools: True
Only weights differ: True
